<a href="https://colab.research.google.com/github/ryougishikifor214/torchcode/blob/master/templates/25_flash_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/25_flash_attention.ipynb)

# 🔴 Hard: Flash Attention (Tiled)

Implement **tiled attention with online softmax** — the core idea behind Flash Attention.

### Signature
```python
def flash_attention(Q, K, V, block_size=32) -> Tensor:
    # Q, K, V: (B, S, D)
    # Returns: (B, S, D) — same as standard attention
```

### Key Insight
Instead of materializing the full S×S attention matrix, process in blocks:
1. For each Q-block, iterate over K/V blocks
2. Use **online softmax**: track running `max` and `sum`
3. Rescale accumulator when max changes: `acc *= exp(old_max - new_max)`
4. Final: `output = acc / row_sum`

Must give **identical** results to standard softmax attention.

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.4 MB/s eta 0:00:00


In [ ]:
def flash_attention(Q, K, V, block_size=32):
    """
    Flash Attention 的简化实现 - 分块计算 + 在线 Softmax
    避免显式构建完整的 SxS 注意力矩阵，内存复杂度从 O(S²) 降到 O(S)

    Args:
        Q: (B, S, D) - 查询矩阵
        K: (B, S, D) - 键矩阵
        V: (B, S, D) - 值矩阵
        block_size: 分块大小，控制内存使用和计算粒度

    Returns:
        output: (B, S, D) - 注意力输出
    """
    B, S, D = Q.shape
    output = torch.zeros_like(Q)  # 初始化输出张量

    # ============ 外循环：遍历 Query 块 ============
    # 每次处理 block_size 个 Query，逐步构建最终输出
    for i in range(0, S, block_size):
        qi = Q[:, i:i+block_size]  # (B, bs_q, D) 当前 Query 块
        bs_q = qi.shape[1]         # 实际 Query 块大小（最后一块可能小于 block_size）

        # ----- 初始化当前 Query 块的统计量 -----
        # row_max: 每个 query 位置当前看到的最大分数，初始为 -inf（数值稳定用）
        row_max = torch.full((B, bs_q, 1), float('-inf'), device=Q.device)

        # row_sum: 每个 query 位置的 exp 之和（softmax 分母），初始为 0
        row_sum = torch.zeros(B, bs_q, 1, device=Q.device)

        # acc: 累加器，存储 exp(分数) * V 的加权和（softmax 分子），初始为 0
        acc = torch.zeros(B, bs_q, D, device=Q.device)

        # ============ 内循环：遍历 Key/Value 块 ============
        # 对每一个 Query 块，扫描所有 Key/Value 块，逐步累积 softmax 结果
        for j in range(0, S, block_size):
            kj = K[:, j:j+block_size]  # (B, bs_k, D) 当前 Key 块
            vj = V[:, j:j+block_size]  # (B, bs_k, D) 当前 Value 块

            # ----- Step 1: 计算当前块的注意力分数 -----
            # Q_block @ K_block^T / sqrt(D)
            # (B, bs_q, D) @ (B, D, bs_k) -> (B, bs_q, bs_k)
            scores = torch.bmm(qi, kj.transpose(1, 2)) / math.sqrt(D)

            # ----- Step 2: 获取当前块的最大值 -----
            # 每个 Query 在 Key 维度上的最大值 (B, bs_q, 1)
            block_max = scores.max(dim=-1, keepdim=True).values

            # ----- Step 3: 更新全局最大值（在线 Softmax 核心）-----
            # 融合历史最大值和当前块最大值，得到新的全局最大值
            # 这是 Flash Attention 的关键：最大值可能被新块更新
            new_max = torch.maximum(row_max, block_max)  # (B, bs_q, 1)

            # ----- Step 4: 计算修正因子（缩放历史值）-----
            # 当全局最大值变大时，需要缩放之前累加的 exp 值
            # 数学原理：exp(x - new_max) = exp(x - row_max) * exp(row_max - new_max)
            # 所以修正因子 = exp(row_max - new_max)
            correction = torch.exp(row_max - new_max)  # (B, bs_q, 1)

            # ----- Step 5: 计算当前块的 exp（数值稳定）-----
            # 用新的全局最大值归一化，确保 exp 最大值为 1，防止溢出
            # 所有块使用同一基准，所以可以安全累加
            exp_scores = torch.exp(scores - new_max)  # (B, bs_q, bs_k)

            # ----- Step 6: 更新累加器（分子）-----
            # acc = acc_old * correction + exp_scores @ V_block
            # 第一项：缩放历史的加权和（因为 max 变了）
            # 第二项：当前块的加权和
            # (B, bs_q, bs_k) @ (B, bs_k, D) -> (B, bs_q, D)
            acc = acc * correction + torch.bmm(exp_scores, vj)

            # ----- Step 7: 更新行和（分母）-----
            # row_sum = row_sum_old * correction + sum(exp_scores)
            # 第一项：缩放历史 exp 之和
            # 第二项：当前块 exp 之和（在 Key 维度求和）
            row_sum = row_sum * correction + exp_scores.sum(dim=-1, keepdim=True)

            # ----- Step 8: 更新最大值 -----
            # 将当前最大值设为新的全局最大值，供下一轮使用
            row_max = new_max

        # ----- 输出归一化 -----
        # acc / row_sum = softmax(scores) @ V
        # 因为：acc = sum(exp(scores - row_max) * V)
        #      row_sum = sum(exp(scores - row_max))
        #      所以 acc / row_sum = softmax(scores) @ V
        # 将当前 Query 块的结果写入输出对应位置
        output[:, i:i+block_size] = acc / row_sum

    return output

In [5]:
import torch
import math
from torch_judge import hint
hint("flash_attention")


💡 Hint for Flash Attention (Tiled):
   Process Q in blocks. For each Q-block, iterate over K/V blocks. Use online softmax: track running max and sum, rescale accumulator when max changes. output = acc / row_sum.



In [6]:
# ✏️ YOUR IMPLEMENTATION HERE

def flash_attention(Q, K, V, block_size=32):
    # Process Q in blocks, iterate K/V blocks with online softmax
    # pass
    B,S,D=Q.shape
    output=torch.zeros_like(Q)

    for i in range(0,S,block_size):
      qi=Q[:,i:i+block_size]
      bs_qi=qi.shape[1]
      row_max=torch.full((B,bs_qi,1),float("-inf"),device=Q.device)
      row_sum=torch.zeros([B,bs_qi,1],device=Q.device)
      acc=torch.zeros([B,bs_qi,D],device=Q.device)
      for j in range(0,S,block_size):
        kj,vj=K[:,j:j+block_size],V[:,j:j+block_size]
        scores=torch.bmm(qi,kj.transpose(-1,-2))/math.sqrt(D)
        block_max=scores.max(dim=-1,keepdim=True).values
        new_max=torch.maximum(row_max,block_max)
        correction=torch.exp(row_max-new_max)
        block_exp_scores=torch.exp(scores-new_max)

        acc=correction*acc+torch.bmm(block_exp_scores,vj)
        row_sum=correction*row_sum+block_exp_scores.sum(dim=-1,keepdim=True)
        row_max=new_max

      output[:,i:i+block_size]=acc/row_sum

    return output

In [7]:
# 🧪 Debug
import math
Q, K, V = torch.randn(1, 8, 4), torch.randn(1, 8, 4), torch.randn(1, 8, 4)
out = flash_attention(Q, K, V, block_size=4)
scores = torch.bmm(Q, K.transpose(1,2)) / math.sqrt(4)
ref = torch.bmm(torch.softmax(scores, dim=-1), V)
print('Match:', torch.allclose(out, ref, atol=1e-4))

Match: True


In [8]:
# ✅ SUBMIT
from torch_judge import check
check('flash_attention')


🧪 Testing: Flash Attention (Tiled) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Matches standard attention (31.8ms)
  ✅ [2/4] Non-aligned block size (2.8ms)
  ✅ [3/4] Block size invariant (2.8ms)
  ✅ [4/4] Gradient flow (43.1ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (80.5ms total)
  Progress saved. Run status() to see your dashboard.

